Prepare data for models

In [107]:
import os
import librosa
import numpy as np
from tqdm import tqdm

DATA_PATH = '../Data/speech_commands/'
SAVE_PATH = '../Data/'

TARGET_WORDS = ['on', 'off', 'stop']
UNKNOWN_WORDS = ['right', 'left', 'up', 'down']

X = [] 
y = [] 

def get_mfcc(file_path):
    try:
        # 1. Load audio file
        audio, sr = librosa.load(file_path, sr=16000)
        
        # 2. FEATURE ENGINEERING: Trim silence from beginning and end
        # top_db=25 means any sound 25dB below the peak volume is considered silence and removed
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=25)
        
        # Filter out extremely short noises (e.g., clicks shorter than 0.15 seconds)
        if len(trimmed_audio) < 2400: 
            return None

        # 3. Extract MFCC on the PURE spoken word
        mfccs = librosa.feature.mfcc(y=trimmed_audio, sr=sr, n_mfcc=13, hop_length=512)
        mfccs = mfccs.T # Transpose to shape (frames, 13)

        # 4. FEATURE ENGINEERING: Dynamic Temporal Pooling
        # Dynamically split the frames into 3 equal sections, regardless of length
        parts = np.array_split(mfccs, 3)
        
        # Calculate the mean for each dynamic part
        part1 = np.mean(parts[0], axis=0) # Always the start of the word
        part2 = np.mean(parts[1], axis=0) # Always the middle
        part3 = np.mean(parts[2], axis=0) # Always the end

        # Concatenate them into a single 1D array of 39 features
        features = np.concatenate((part1, part2, part3))
        
        return features
    except Exception as e:
        return None

print("Processing Target Words (with Silence Trimming)...")
for label_idx, word in enumerate(TARGET_WORDS):
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        print(f"Warning: Directory for '{word}' not found.")
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:1500], desc=f"Loading {word}"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(label_idx)

print("\nProcessing Unknown Words (with Silence Trimming)...")
unknown_label_idx = len(TARGET_WORDS) 
for word in UNKNOWN_WORDS:
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:375], desc=f"Loading {word} (Unknown)"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(unknown_label_idx)

X = np.array(X)
y = np.array(y)

np.save(os.path.join(SAVE_PATH, 'X_features.npy'), X)
np.save(os.path.join(SAVE_PATH, 'y_labels.npy'), y)

print(f"\nFeature extraction complete! Saved {len(X)} robust samples.")
print(f"NEW Features shape: {X.shape} (Dynamic Temporal Pooling)")

Processing Target Words (with Silence Trimming)...


Loading stop: 100%|██████████| 1500/1500 [00:06<00:00, 228.39it/s]



Processing Unknown Words (with Silence Trimming)...


Loading down (Unknown): 100%|██████████| 375/375 [00:01<00:00, 236.07it/s]


Feature extraction complete! Saved 5995 robust samples.
NEW Features shape: (5995, 39) (Dynamic Temporal Pooling)


RandomForest

In [98]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
# import m2cgen as m2c
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define the Optuna objective function for a TINY Random Forest model
def objective(trial):
    param = {
        # STRICT LIMITS: Keeping trees small for the MCU's RAM
        'n_estimators': trial.suggest_int('n_estimators', 10, 30), 
        'max_depth': trial.suggest_int('max_depth', 2, 10), 
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = RandomForestClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Run the hyperparameter optimization
print("\nStarting Optuna optimization for a lightweight model...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the final Random Forest model
print("\nTraining the final Random Forest model...")
final_model = RandomForestClassifier(**study.best_params, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")

unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print(classification_report(y_test, final_preds, target_names=target_names))

# 5. Export the trained model parameters as lightweight arrays for MicroPython
print("\nExporting tree parameters (arrays) for memory-efficient Edge Deployment...")

trees_dump = []
for estimator in final_model.estimators_:
    tree = estimator.tree_
    
    # In Random Forest, tree.value stores the class counts at each node
    # We convert this to the actual predicted class index for leaf nodes
    node_classes = [int(np.argmax(val)) for val in tree.value]
    
    tree_data = {
        'left': tree.children_left.tolist(),
        'right': tree.children_right.tolist(),
        'feature': tree.feature.tolist(),
        # Round thresholds to 4 decimals to save storage space
        'threshold': [round(t, 4) for t in tree.threshold.tolist()], 
        'classes': node_classes
    }
    trees_dump.append(tree_data)

# Save to a Python file as a simple list of dictionaries
output_file = os.path.join(OUTPUT_PATH, 'model_data_randomForest.py')
with open(output_file, 'w') as f:
    f.write("# Auto-generated Random Forest parameters for Edge AI\n")
    f.write("TREES = [\n")
    for t in trees_dump:
        f.write(f"    {t},\n")
    f.write("]\n")

print(f"Model parameters successfully exported to: {output_file}")

[I 2026-06-30 17:00:36,811] A new study created in memory with name: no-name-b4b133dd-03fe-435b-967c-5ce238ef39d1


Loaded Data -> X shape: (5995, 39), y shape: (5995,)

Starting Optuna optimization for a lightweight model...


[I 2026-06-30 17:00:37,159] Trial 0 finished with value: 0.7222685571309424 and parameters: {'n_estimators': 17, 'max_depth': 5}. Best is trial 0 with value: 0.7222685571309424.
[I 2026-06-30 17:00:37,322] Trial 1 finished with value: 0.718932443703086 and parameters: {'n_estimators': 29, 'max_depth': 5}. Best is trial 0 with value: 0.7222685571309424.
[I 2026-06-30 17:00:37,455] Trial 2 finished with value: 0.755629691409508 and parameters: {'n_estimators': 27, 'max_depth': 7}. Best is trial 2 with value: 0.755629691409508.
[I 2026-06-30 17:00:37,526] Trial 3 finished with value: 0.6788990825688074 and parameters: {'n_estimators': 13, 'max_depth': 4}. Best is trial 2 with value: 0.755629691409508.
[I 2026-06-30 17:00:37,672] Trial 4 finished with value: 0.755629691409508 and parameters: {'n_estimators': 27, 'max_depth': 7}. Best is trial 2 with value: 0.755629691409508.
[I 2026-06-30 17:00:37,761] Trial 5 finished with value: 0.6572143452877398 and parameters: {'n_estimators': 19, 'ma


Best parameters found: {'n_estimators': 28, 'max_depth': 9}
Best cross-validation accuracy: 79.15%

Training the final Random Forest model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.78      0.79      0.78       299
         off       0.80      0.80      0.80       300
        stop       0.90      0.84      0.87       300
     unknown       0.70      0.74      0.72       300

    accuracy                           0.79      1199
   macro avg       0.79      0.79      0.79      1199
weighted avg       0.79      0.79      0.79      1199


Exporting tree parameters (arrays) for memory-efficient Edge Deployment...
Model parameters successfully exported to: ../edge_mcu\model_data_randomForest.py


LogisticRegression

In [99]:
import numpy as np
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define the Optuna objective function for Logistic Regression
def objective(trial):
    param = {
        # 'C' is the inverse of regularization strength. 
        # Smaller values specify stronger limits/constraints on the weights.
        'C': trial.suggest_float('C', 1e-4, 1e2, log=True),
        
        # 'solver' is the algorithm used to optimize the weights
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'newton-cg', 'saga']),
        
        # Max_iter is kept high to allow the mathematical solver to fully converge
        'max_iter': 2000, 
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = LogisticRegression(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Run the hyperparameter optimization
print("\nStarting Optuna optimization for Logistic Regression...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the final model using the exact optimized parameters
print("\nTraining the final Logistic Regression model...")
final_model = LogisticRegression(**study.best_params, max_iter=2000, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")

# Dynamically extract classes
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]
print(classification_report(y_test, final_preds, target_names=target_names))

# 5. Export weights and intercepts to a tiny Python file
print("\nExporting model parameters (Weights & Intercepts) for Edge Deployment...")

weights = final_model.coef_.tolist()
intercepts = final_model.intercept_.tolist()

output_file = os.path.join(OUTPUT_PATH, 'model_data_regression.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Lightweight Model for Keyword Spotting\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    # We round the numbers to 4 decimal places to save even more space on the MCU
    f.write("WEIGHTS = [\n")
    for class_weights in weights:
        rounded_weights = [round(w, 4) for w in class_weights]
        f.write(f"    {rounded_weights},\n")
    f.write("]\n\n")
    
    rounded_intercepts = [round(i, 4) for i in intercepts]
    f.write(f"INTERCEPTS = {rounded_intercepts}\n")

print(f"Model successfully exported to: {output_file}")
print("Done! The model_data.py is now highly optimized and incredibly lightweight.")

[I 2026-06-30 17:00:45,942] A new study created in memory with name: no-name-2fef8953-76e1-4963-83ae-daa2ebcb6de7


Loaded Data -> X shape: (5995, 39), y shape: (5995,)

Starting Optuna optimization for Logistic Regression...


c:\Users\User\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
[I 2026-06-30 17:00:47,346] Trial 0 finished with value: 0.7939949958298582 and parameters: {'C': 0.0007574107785051374, 'solver': 'saga'}. Best is trial 0 with value: 0.7939949958298582.
c:\Users\User\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\User\anaconda3\Lib\site-packages\scipy\optimize\_linesearch.py:312: LineSearchWarning: Rounding errors prevent the line search from converging
  alpha_star, phi_star, old_fval, derphi_star = scalar_search_wolfe2(
c:\Users\User\anaconda3\Lib\site-packages\sklearn\utils\optimize.py:100:


Best parameters found: {'C': 0.17486365739236015, 'solver': 'saga'}
Best cross-validation accuracy: 79.48%

Training the final Logistic Regression model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.79      0.82      0.80       299
         off       0.82      0.79      0.80       300
        stop       0.88      0.82      0.85       300
     unknown       0.70      0.75      0.73       300

    accuracy                           0.79      1199
   macro avg       0.80      0.79      0.80      1199
weighted avg       0.80      0.79      0.80      1199


Exporting model parameters (Weights & Intercepts) for Edge Deployment...
Model successfully exported to: ../edge_mcu\model_data_regression.py
Done! The model_data.py is now highly optimized and incredibly lightweight.


In [102]:
import sys
import os

sys.path.append(os.path.abspath('../edge_mcu'))

import model_data_regression as model_data

def predict_audio_class(mfcc_features):
    """
    Computes the dot product of features and weights for each class,
    adds the intercept, and returns the class with the highest score.
    """
    num_classes = len(model_data.INTERCEPTS)
    num_features = len(mfcc_features)
    
    scores = []
    
    # Calculate score for each class
    for i in range(num_classes):
        class_score = model_data.INTERCEPTS[i]
        
        # Dot product: multiply each feature by its corresponding weight
        for j in range(num_features):
            class_score += mfcc_features[j] * model_data.WEIGHTS[i][j]
            
        scores.append(class_score)
        
    # Find the index of the highest score (Argmax)
    best_class_idx = 0
    max_score = scores[0]
    for i in range(1, num_classes):
        if scores[i] > max_score:
            max_score = scores[i]
            best_class_idx = i
            
    return best_class_idx

# --- Example Usage on Board ---
# ['on', 'off', 'stop', 'uknown']
i = 16
features = [-2.5686844e+02,  1.3959177e+02,  8.0665617e+00, -4.1448826e+01, -2.9944046e+01,  3.2988396e-01,  6.2178426e+00,  9.0643892e+00, -6.5838027e+00,  5.9275532e+00, -2.8192373e+01,  1.9626240e+01, -1.0130860e+00, -2.5470102e+02,  1.6171916e+02, -9.4717093e+00, -1.6660915e+01,  9.5316715e+00,  1.5823692e+01, -4.0831656e+00, -1.6109245e+01, -1.6403475e+01, -2.7065601e+00, -7.6615257e+00, 1.2409652e+01, -2.4190521e+01, -4.9443311e+02,  8.9877983e+01, 1.8117041e+01,  3.1036968e+01,  5.0818684e+01,  3.1483164e+01, 4.5603323e+00, -1.2117879e+01, -1.4944232e+01,  2.7241716e+00, 7.9998846e+00, -1.2753278e+01, -1.6924435e+01] # 13 features from mic
result_idx = predict_audio_class(features)
print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i])

IndexError: list index out of range

XGBoost

In [103]:
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os
import json

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define Optuna objective for XGBoost
def objective(trial):
    param = {
        'objective': 'multi:softprob',
        'max_depth': trial.suggest_int('max_depth', 2, 3), 
        'n_estimators': trial.suggest_int('n_estimators', 2, 5), 
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Optimize Hyperparameters
print("\nStarting Optuna optimization for XGBoost...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the Final XGBoost Model
print("\nTraining the final XGBoost model...")
best_params = study.best_params
best_params['objective'] = 'multi:softprob'
best_params['random_state'] = 42

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]
print(classification_report(y_test, final_preds, target_names=target_names))

# 5. CUSTOM EXPORT: Bypass m2cgen and extract trees directly
print("\nExporting XGBoost trees manually for ultra-lightweight Edge Deployment...")

booster = final_model.get_booster()
# Dump all trees to JSON format
trees_json = booster.get_dump(dump_format='json')

parsed_trees = []
for tree_str in trees_json:
    tree_dict = json.loads(tree_str)
    
    # Recursive function to find the maximum node ID to size our arrays
    def get_max_nodeid(node):
        max_id = node.get('nodeid', 0)
        if 'children' in node:
            for child in node['children']:
                max_id = max(max_id, get_max_nodeid(child))
        return max_id
        
    size = get_max_nodeid(tree_dict) + 1
    
    # Initialize flattened arrays for MicroPython
    features = [-1] * size
    thresholds = [0.0] * size
    lefts = [-1] * size
    rights = [-1] * size
    values = [0.0] * size
    
    # Recursive function to parse JSON into flat arrays
    def parse_node(node):
        nid = node['nodeid']
        if 'leaf' in node:
            values[nid] = round(node['leaf'], 4)
        else:
            # XGBoost features are formatted as 'f0', 'f1', etc. We extract the integer.
            feat_str = node['split']
            features[nid] = int(feat_str.replace('f', ''))
            thresholds[nid] = round(node['split_condition'], 4)
            lefts[nid] = node['yes']
            rights[nid] = node['no']
            for child in node['children']:
                parse_node(child)
                
    parse_node(tree_dict)
    
    parsed_trees.append({
        'features': features,
        'thresholds': thresholds,
        'left': lefts,
        'right': rights,
        'values': values
    })

# Save the parsed structure to a Python file
output_file = os.path.join(OUTPUT_PATH, 'model_data_xgb.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Custom XGBoost Export for MicroPython\n\n")
    f.write(f"CLASSES = {target_names}\n")
    f.write(f"NUM_CLASS = {len(target_names)}\n\n")
    f.write("TREES = [\n")
    for t in parsed_trees:
        f.write(f"    {t},\n")
    f.write("]\n")

print(f"XGBoost Model successfully exported to: {output_file}")

[I 2026-06-30 17:03:23,895] A new study created in memory with name: no-name-c84b3a6a-b794-4730-a31d-b8ab24300b68
[I 2026-06-30 17:03:23,972] Trial 0 finished with value: 0.6130108423686406 and parameters: {'max_depth': 2, 'n_estimators': 5, 'learning_rate': 0.044450120030737617}. Best is trial 0 with value: 0.6130108423686406.
[I 2026-06-30 17:03:24,017] Trial 1 finished with value: 0.6805671392827356 and parameters: {'max_depth': 3, 'n_estimators': 4, 'learning_rate': 0.244055553256302}. Best is trial 1 with value: 0.6805671392827356.
[I 2026-06-30 17:03:24,045] Trial 2 finished with value: 0.6130108423686406 and parameters: {'max_depth': 2, 'n_estimators': 3, 'learning_rate': 0.20427392546223994}. Best is trial 1 with value: 0.6805671392827356.
[I 2026-06-30 17:03:24,066] Trial 3 finished with value: 0.6021684737281068 and parameters: {'max_depth': 2, 'n_estimators': 2, 'learning_rate': 0.24615402138846437}. Best is trial 1 with value: 0.6805671392827356.
[I 2026-06-30 17:03:24,094]

Loaded Data -> X shape: (5995, 39), y shape: (5995,)

Starting Optuna optimization for XGBoost...


[I 2026-06-30 17:03:24,150] Trial 5 finished with value: 0.6547122602168474 and parameters: {'max_depth': 2, 'n_estimators': 5, 'learning_rate': 0.2395100718412971}. Best is trial 1 with value: 0.6805671392827356.
[I 2026-06-30 17:03:24,196] Trial 6 finished with value: 0.683069224353628 and parameters: {'max_depth': 3, 'n_estimators': 4, 'learning_rate': 0.24900175132385646}. Best is trial 6 with value: 0.683069224353628.
[I 2026-06-30 17:03:24,234] Trial 7 finished with value: 0.6130108423686406 and parameters: {'max_depth': 2, 'n_estimators': 5, 'learning_rate': 0.04995279777706157}. Best is trial 6 with value: 0.683069224353628.
[I 2026-06-30 17:03:24,271] Trial 8 finished with value: 0.6563803169307757 and parameters: {'max_depth': 2, 'n_estimators': 5, 'learning_rate': 0.28808477659464676}. Best is trial 6 with value: 0.683069224353628.
[I 2026-06-30 17:03:24,305] Trial 9 finished with value: 0.6755629691409508 and parameters: {'max_depth': 3, 'n_estimators': 3, 'learning_rate': 


Best parameters found: {'max_depth': 3, 'n_estimators': 4, 'learning_rate': 0.24900175132385646}
Best cross-validation accuracy: 68.31%

Training the final XGBoost model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.65      0.71      0.68       299
         off       0.75      0.70      0.73       300
        stop       0.83      0.67      0.74       300
     unknown       0.56      0.65      0.60       300

    accuracy                           0.68      1199
   macro avg       0.70      0.68      0.69      1199
weighted avg       0.70      0.68      0.69      1199


Exporting XGBoost trees manually for ultra-lightweight Edge Deployment...
XGBoost Model successfully exported to: ../edge_mcu\model_data_xgb.py


In [104]:
import sys
import os

sys.path.append(os.path.abspath('../edge_mcu'))

import model_data_xgb as model_data
def predict_audio_class(mfcc_features):
    """
    Traverses the exported XGBoost trees using the extracted MFCC features.
    Returns the index of the predicted class based on the highest raw score.
    """
    num_class = model_data.NUM_CLASS
    
    # Array to accumulate scores (raw logits) for each class
    class_scores = [0.0] * num_class
    
    # In multi-class XGBoost, trees are assigned to classes in a round-robin fashion.
    # Tree 0 -> Class 0, Tree 1 -> Class 1 ... Tree 4 -> Class 0, etc.
    for i, tree in enumerate(model_data.TREES):
        node = 0 # Start at the root of the tree
        
        # Traverse until a leaf is reached (marked by feature index -1)
        while tree['features'][node] != -1:
            feat_idx = tree['features'][node]
            
            # XGBoost splitting rule: strictly less than (<)
            if mfcc_features[feat_idx] < tree['thresholds'][node]:
                node = tree['left'][node]
            else:
                node = tree['right'][node]
                
        # Leaf reached: get the value and add it to the corresponding class score
        leaf_value = tree['values'][node]
        class_idx = i % num_class
        class_scores[class_idx] += leaf_value
        
    # Find the index of the highest score (Argmax)
    best_class = 0
    max_score = class_scores[0]
    for i in range(1, num_class):
        if class_scores[i] > max_score:
            max_score = class_scores[i]
            best_class = i
            
    return best_class

# --- Example Usage on Board ---
# ['on', 'off', 'stop', 'uknown']
i = 16
features = [-2.5686844e+02,  1.3959177e+02,  8.0665617e+00, -4.1448826e+01, -2.9944046e+01,  3.2988396e-01,  6.2178426e+00,  9.0643892e+00, -6.5838027e+00,  5.9275532e+00, -2.8192373e+01,  1.9626240e+01, -1.0130860e+00, -2.5470102e+02,  1.6171916e+02, -9.4717093e+00, -1.6660915e+01,  9.5316715e+00,  1.5823692e+01, -4.0831656e+00, -1.6109245e+01, -1.6403475e+01, -2.7065601e+00, -7.6615257e+00, 1.2409652e+01, -2.4190521e+01, -4.9443311e+02,  8.9877983e+01, 1.8117041e+01,  3.1036968e+01,  5.0818684e+01,  3.1483164e+01, 4.5603323e+00, -1.2117879e+01, -1.4944232e+01,  2.7241716e+00, 7.9998846e+00, -1.2753278e+01, -1.6924435e+01] # 13 features from mic
result_idx = predict_audio_class(features)
print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i])


Predicted Word: off and actuall is: 1


*now with Root reduction and feature importance*

In [105]:
import os
import librosa
import numpy as np
from tqdm import tqdm

DATA_PATH = '../Data/speech_commands/'
SAVE_PATH = '../Data/'

TARGET_WORDS = ['on', 'off', 'stop']
UNKNOWN_WORDS = ['right', 'left', 'up', 'down']

X = [] 
y = [] 

def get_mfcc(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=16000)
        
        # Trim silence
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=25)
        if len(trimmed_audio) < 2400: 
            return None

        # ROOT REDUCTION: Extract only 8 MFCCs instead of 13
        mfccs = librosa.feature.mfcc(y=trimmed_audio, sr=sr, n_mfcc=8, hop_length=512)
        mfccs = mfccs.T 

        # Dynamic Temporal Pooling (3 parts)
        parts = np.array_split(mfccs, 3)
        
        part1 = np.mean(parts[0], axis=0) 
        part2 = np.mean(parts[1], axis=0) 
        part3 = np.mean(parts[2], axis=0) 

        # Concatenate: 8 + 8 + 8 = 24 features
        features = np.concatenate((part1, part2, part3))
        
        return features
    except Exception as e:
        return None

print("Processing Target Words (with 8-MFCC Root Reduction)...")
for label_idx, word in enumerate(TARGET_WORDS):
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:1500], desc=f"Loading {word}"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(label_idx)

print("\nProcessing Unknown Words...")
unknown_label_idx = len(TARGET_WORDS) 
for word in UNKNOWN_WORDS:
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:375], desc=f"Loading {word} (Unknown)"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(unknown_label_idx)

X = np.array(X)
y = np.array(y)

np.save(os.path.join(SAVE_PATH, 'X_features.npy'), X)
np.save(os.path.join(SAVE_PATH, 'y_labels.npy'), y)

print(f"\nFeature extraction complete! Saved {len(X)} samples.")
print(f"NEW Features shape: {X.shape} (24 features per sample)")

Processing Target Words (with 8-MFCC Root Reduction)...


Loading stop: 100%|██████████| 1500/1500 [00:05<00:00, 254.91it/s]



Processing Unknown Words...


Loading down (Unknown): 100%|██████████| 375/375 [00:01<00:00, 243.86it/s]


Feature extraction complete! Saved 5995 samples.
NEW Features shape: (5995, 24) (24 features per sample)


In [106]:
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os
import json

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load Data
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))
print(f"Original Data -> X shape: {X.shape}, y shape: {y.shape}")

X_train_full, X_test_full, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. FEATURE SELECTION (Method 1)
print("\n--- Running Feature Selection ---")
# Train a temporary, unconstrained model to find feature importance
temp_model = xgb.XGBClassifier(objective='multi:softprob', random_state=42)
temp_model.fit(X_train_full, y_train)

# Get importance scores and select the top 12 features (out of 24)
importances = temp_model.feature_importances_
# argsort sorts ascending, so we take the last 12 elements for the highest scores
top_12_indices = np.argsort(importances)[-12:] 
top_12_indices = np.sort(top_12_indices) # Sort indices to keep the original order

print(f"Selected Top 12 Feature Indices: {top_12_indices.tolist()}")

# Filter the datasets to ONLY include these 12 features
X_train = X_train_full[:, top_12_indices]
X_test = X_test_full[:, top_12_indices]
print(f"Reduced Data -> X_train shape: {X_train.shape}")

# 3. OPTUNA OPTIMIZATION (on the reduced 12-feature dataset)
print("\n--- Starting Optuna optimization on reduced features ---")
def objective(trial):
    param = {
        'objective': 'multi:softprob',
        # Keep trees extremely shallow for Edge RAM
        'max_depth': trial.suggest_int('max_depth', 2, 3), 
        'n_estimators': trial.suggest_int('n_estimators', 2, 5), 
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"\nBest Optuna Accuracy: {study.best_value * 100:.2f}%")

# 4. TRAIN FINAL MODEL
print("\n--- Training Final Lightweight XGBoost ---")
best_params = study.best_params
best_params['objective'] = 'multi:softprob'
best_params['random_state'] = 42

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

preds = final_model.predict(X_test)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]
print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 5. CUSTOM EXPORT TO PYTHON ARRAYS
print("\n--- Exporting arrays to model_data.py ---")
booster = final_model.get_booster()
trees_json = booster.get_dump(dump_format='json')

parsed_trees = []
for tree_str in trees_json:
    tree_dict = json.loads(tree_str)
    
    def get_max_nodeid(node):
        max_id = node.get('nodeid', 0)
        if 'children' in node:
            for child in node['children']:
                max_id = max(max_id, get_max_nodeid(child))
        return max_id
        
    size = get_max_nodeid(tree_dict) + 1
    features = [-1] * size
    thresholds = [0.0] * size
    lefts = [-1] * size
    rights = [-1] * size
    values = [0.0] * size
    
    def parse_node(node):
        nid = node['nodeid']
        if 'leaf' in node:
            values[nid] = round(node['leaf'], 4)
        else:
            feat_str = node['split']
            features[nid] = int(feat_str.replace('f', ''))
            thresholds[nid] = round(node['split_condition'], 4)
            lefts[nid] = node['yes']
            rights[nid] = node['no']
            for child in node['children']:
                parse_node(child)
                
    parse_node(tree_dict)
    
    parsed_trees.append({
        'features': features,
        'thresholds': thresholds,
        'left': lefts,
        'right': rights,
        'values': values
    })

output_file = os.path.join(OUTPUT_PATH, 'model_data_xgb_new.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Optimized Custom XGBoost Export\n\n")
    f.write(f"CLASSES = {target_names}\n")
    f.write(f"NUM_CLASS = {len(target_names)}\n")
    # We MUST save the selected feature indices so the MCU knows which features to pass to the model!
    f.write(f"SELECTED_FEATURES = {top_12_indices.tolist()}\n\n") 
    f.write("TREES = [\n")
    for t in parsed_trees:
        f.write(f"    {t},\n")
    f.write("]\n")

print(f"Extremely lightweight XGBoost successfully exported to: {output_file}")

Original Data -> X shape: (5995, 24), y shape: (5995,)

--- Running Feature Selection ---


[I 2026-06-30 17:04:23,465] A new study created in memory with name: no-name-4f732bbb-ab6b-4ecd-950f-cff733bf1c84
[I 2026-06-30 17:04:23,483] Trial 0 finished with value: 0.6113427856547122 and parameters: {'max_depth': 2, 'n_estimators': 3, 'learning_rate': 0.2068086679372707}. Best is trial 0 with value: 0.6113427856547122.
[I 2026-06-30 17:04:23,513] Trial 1 finished with value: 0.6763969974979149 and parameters: {'max_depth': 3, 'n_estimators': 5, 'learning_rate': 0.2209466472893033}. Best is trial 1 with value: 0.6763969974979149.
[I 2026-06-30 17:04:23,535] Trial 2 finished with value: 0.646371976647206 and parameters: {'max_depth': 2, 'n_estimators': 5, 'learning_rate': 0.14601863698763198}. Best is trial 1 with value: 0.6763969974979149.
[I 2026-06-30 17:04:23,555] Trial 3 finished with value: 0.6630525437864887 and parameters: {'max_depth': 3, 'n_estimators': 3, 'learning_rate': 0.2572445969395973}. Best is trial 1 with value: 0.6763969974979149.
[I 2026-06-30 17:04:23,576] Tr

Selected Top 12 Feature Indices: [1, 3, 4, 6, 9, 11, 14, 15, 17, 19, 20, 22]
Reduced Data -> X_train shape: (4796, 12)

--- Starting Optuna optimization on reduced features ---


[I 2026-06-30 17:04:23,685] Trial 9 finished with value: 0.6722268557130943 and parameters: {'max_depth': 3, 'n_estimators': 5, 'learning_rate': 0.21102300067438834}. Best is trial 1 with value: 0.6763969974979149.
[I 2026-06-30 17:04:23,715] Trial 10 finished with value: 0.6522101751459549 and parameters: {'max_depth': 3, 'n_estimators': 4, 'learning_rate': 0.12170452982920332}. Best is trial 1 with value: 0.6763969974979149.
[I 2026-06-30 17:04:23,751] Trial 11 finished with value: 0.6688907422852377 and parameters: {'max_depth': 3, 'n_estimators': 5, 'learning_rate': 0.16627486078283654}. Best is trial 1 with value: 0.6763969974979149.
[I 2026-06-30 17:04:23,786] Trial 12 finished with value: 0.646371976647206 and parameters: {'max_depth': 3, 'n_estimators': 5, 'learning_rate': 0.031063631941615766}. Best is trial 1 with value: 0.6763969974979149.
[I 2026-06-30 17:04:23,821] Trial 13 finished with value: 0.6797331109257715 and parameters: {'max_depth': 3, 'n_estimators': 5, 'learnin


Best Optuna Accuracy: 69.22%

--- Training Final Lightweight XGBoost ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.64      0.69      0.67       299
         off       0.75      0.73      0.74       300
        stop       0.84      0.70      0.77       300
     unknown       0.57      0.64      0.61       300

    accuracy                           0.69      1199
   macro avg       0.70      0.69      0.70      1199
weighted avg       0.70      0.69      0.70      1199


--- Exporting arrays to model_data.py ---
Extremely lightweight XGBoost successfully exported to: ../edge_mcu\model_data_xgb_new.py


*Gaussian Naive Bayes*

In [108]:
import numpy as np
from sklearn.naive_bayes import GaussianNB
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'

# 1. Load the FULL 39-feature dataset
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. OPTUNA OPTIMIZATION FOR NAIVE BAYES
print("\n--- Starting Optuna optimization for var_smoothing ---")

def objective(trial):
    # Suggest a value for var_smoothing on a log scale (from very small to relatively large)
    var_smoothing = trial.suggest_float('var_smoothing', 1e-11, 1e-1, log=True)
    
    gnb = GaussianNB(var_smoothing=var_smoothing)
    gnb.fit(X_train, y_train)
    preds = gnb.predict(X_test)
    
    return accuracy_score(y_test, preds)

study = optuna.create_study(direction='maximize')
# GNB is incredibly fast, so we can easily run 100 trials in seconds!
study.optimize(objective, n_trials=100) 

print(f"\nBest Optuna var_smoothing: {study.best_params['var_smoothing']}")
print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")

# 3. TRAIN FINAL MODEL WITH BEST PARAMETERS
print("\n--- Training Final Optimized Gaussian Naive Bayes ---")
final_gnb = GaussianNB(**study.best_params)
final_gnb.fit(X_train, y_train)

preds = final_gnb.predict(X_test)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 4. EXPORT TO PURE PYTHON ARRAYS (Same size, higher accuracy!)
print("\n--- Exporting Statistical Parameters to model_data.py ---")

class_priors = final_gnb.class_prior_
if class_priors is None:
    class_counts = np.bincount(y_train)
    class_priors = class_counts / len(y_train)

priors_list = class_priors.tolist()
means_list = final_gnb.theta_.tolist()
vars_list = final_gnb.var_.tolist()

output_file = os.path.join(DATA_PATH, 'model_data.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Optuna-Optimized Gaussian Naive Bayes Export\n\n")
    f.write(f"CLASSES = {target_names}\n")
    
    pure_priors = [round(float(p), 6) for p in priors_list]
    f.write(f"PRIORS = {pure_priors}\n\n")
    
    f.write("MEANS = [\n")
    for class_means in means_list:
        pure_means = [round(float(m), 6) for m in class_means]
        f.write(f"    {pure_means},\n")
    f.write("]\n\n")
    
    # Notice we don't need to manually add epsilon here anymore, 
    # the optimal var_smoothing already handled the math stabilization!
    f.write("VARIANCES = [\n")
    for class_vars in vars_list:
        pure_vars = [round(float(v), 8) for v in class_vars]
        f.write(f"    {pure_vars},\n")
    f.write("]\n")

print(f"Optimized Model successfully exported to: {output_file}")

[I 2026-06-30 17:05:23,096] A new study created in memory with name: no-name-d8d670ed-d447-46d1-b7ec-982f3361e698
[I 2026-06-30 17:05:23,102] Trial 0 finished with value: 0.7306088407005839 and parameters: {'var_smoothing': 0.00030986812476685216}. Best is trial 0 with value: 0.7306088407005839.


[I 2026-06-30 17:05:23,106] Trial 1 finished with value: 0.7306088407005839 and parameters: {'var_smoothing': 2.1594503266402378e-11}. Best is trial 0 with value: 0.7306088407005839.
[I 2026-06-30 17:05:23,111] Trial 2 finished with value: 0.7306088407005839 and parameters: {'var_smoothing': 8.901883045089085e-06}. Best is trial 0 with value: 0.7306088407005839.
[I 2026-06-30 17:05:23,115] Trial 3 finished with value: 0.7214345287739783 and parameters: {'var_smoothing': 0.005700225484334213}. Best is trial 0 with value: 0.7306088407005839.
[I 2026-06-30 17:05:23,120] Trial 4 finished with value: 0.7306088407005839 and parameters: {'var_smoothing': 2.9114216539249266e-11}. Best is trial 0 with value: 0.7306088407005839.
[I 2026-06-30 17:05:23,125] Trial 5 finished with value: 0.7306088407005839 and parameters: {'var_smoothing': 0.0005200110670670832}. Best is trial 0 with value: 0.7306088407005839.
[I 2026-06-30 17:05:23,130] Trial 6 finished with value: 0.7306088407005839 and parameter

Loaded Data -> X shape: (5995, 39), y shape: (5995,)

--- Starting Optuna optimization for var_smoothing ---


[I 2026-06-30 17:05:23,297] Trial 34 finished with value: 0.7289407839866555 and parameters: {'var_smoothing': 0.0007317684255032323}. Best is trial 18 with value: 0.731442869057548.
[I 2026-06-30 17:05:23,302] Trial 35 finished with value: 0.7306088407005839 and parameters: {'var_smoothing': 7.07619666494358e-06}. Best is trial 18 with value: 0.731442869057548.
[I 2026-06-30 17:05:23,308] Trial 36 finished with value: 0.7297748123436196 and parameters: {'var_smoothing': 1.098965187346719e-07}. Best is trial 18 with value: 0.731442869057548.
[I 2026-06-30 17:05:23,314] Trial 37 finished with value: 0.7306088407005839 and parameters: {'var_smoothing': 0.00037514593288151765}. Best is trial 18 with value: 0.731442869057548.
[I 2026-06-30 17:05:23,319] Trial 38 finished with value: 0.7289407839866555 and parameters: {'var_smoothing': 0.0022153027626991887}. Best is trial 18 with value: 0.731442869057548.
[I 2026-06-30 17:05:23,325] Trial 39 finished with value: 0.7114261884904087 and para


Best Optuna var_smoothing: 0.00012084666167390031
Best Accuracy during CV: 73.14%

--- Training Final Optimized Gaussian Naive Bayes ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.72      0.74      0.73       299
         off       0.79      0.71      0.75       300
        stop       0.80      0.80      0.80       300
     unknown       0.64      0.67      0.65       300

    accuracy                           0.73      1199
   macro avg       0.73      0.73      0.73      1199
weighted avg       0.73      0.73      0.73      1199


--- Exporting Statistical Parameters to model_data.py ---
Optimized Model successfully exported to: ../Data/model_data.py


In [109]:
import math
import sys
import os

sys.path.append(os.path.abspath('../edge_mcu'))

import model_data_gnb as model_data
def predict_audio_class(mfcc_features):
    """
    Computes the Gaussian Log-Probability for each class and returns
    the index of the class with the highest probability.
    """
    best_class = 0
    # Initialize with negative infinity
    max_log_prob = -float('inf') 
    
    num_classes = len(model_data.CLASSES)
    num_features = len(mfcc_features)
    
    for c in range(num_classes):
        # Start with the log of the prior probability for this class
        log_prob = math.log(model_data.PRIORS[c])
        
        # Add the log probability of each feature given the class
        for i in range(num_features):
            x = mfcc_features[i]
            mean = model_data.MEANS[c][i]
            var = model_data.VARIANCES[c][i]
            
            # Gaussian Log-PDF Formula: 
            # -0.5 * log(variance) - 0.5 * ((x - mean)^2 / variance)
            # (We drop the -0.5 * log(2*pi) constant as it's the same for all classes)
            
            term1 = math.log(var)
            term2 = ((x - mean) ** 2) / var
            
            log_prob -= 0.5 * (term1 + term2)
            
        # Keep track of the highest probability
        if log_prob > max_log_prob:
            max_log_prob = log_prob
            best_class = c
            
    return best_class

# --- Example Usage on Board ---
# features = [1.2, -0.5, 3.4, ...] # The 39 MFCC features extracted from the mic
# result_idx = predict_audio_class(features)
# print("Predicted Word:", model_data.CLASSES[result_idx])


# --- Example Usage on Board ---
# ['on', 'off', 'stop', 'uknown']
i = 16
# features = [-2.5686844e+02,  1.3959177e+02,  8.0665617e+00, -4.1448826e+01, -2.9944046e+01,  3.2988396e-01,  6.2178426e+00,  9.0643892e+00, -6.5838027e+00,  5.9275532e+00, -2.8192373e+01,  1.9626240e+01, -1.0130860e+00, -2.5470102e+02,  1.6171916e+02, -9.4717093e+00, -1.6660915e+01,  9.5316715e+00,  1.5823692e+01, -4.0831656e+00, -1.6109245e+01, -1.6403475e+01, -2.7065601e+00, -7.6615257e+00, 1.2409652e+01, -2.4190521e+01, -4.9443311e+02,  8.9877983e+01, 1.8117041e+01,  3.1036968e+01,  5.0818684e+01,  3.1483164e+01, 4.5603323e+00, -1.2117879e+01, -1.4944232e+01,  2.7241716e+00, 7.9998846e+00, -1.2753278e+01, -1.6924435e+01] # 13 features from mic
import random
for j in range(10):
    i = random.randint(0, len(y_test))
    features = X_test[i]
    result_idx = predict_audio_class(features)
    print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i] + 1)

Predicted Word: on and actuall is: 1
Predicted Word: stop and actuall is: 2
Predicted Word: stop and actuall is: 3
Predicted Word: unknown and actuall is: 3
Predicted Word: on and actuall is: 1
Predicted Word: unknown and actuall is: 4
Predicted Word: off and actuall is: 2
Predicted Word: off and actuall is: 3
Predicted Word: off and actuall is: 2
Predicted Word: stop and actuall is: 3


*LDA + Logistic regression*

In [110]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load the FULL 39-feature dataset
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. STAGE 1: Dimensionality Reduction using LDA
print("\n--- Running LDA to reduce 39 features to 3 Super-Features ---")
# n_components is always (number of classes - 1) for LDA
lda = LinearDiscriminantAnalysis(n_components=3)
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda = lda.transform(X_test)

print(f"New Training Data Shape after LDA: {X_train_lda.shape}")

# 3. STAGE 2: Optuna Optimization for Logistic Regression
print("\n--- Starting Optuna optimization for Logistic Regression ---")
def objective(trial):
    # Tune the regularization parameter C
    c_val = trial.suggest_float('C', 1e-4, 1e2, log=True)
    
    lr = LogisticRegression(C=c_val, max_iter=5000, random_state=42)
    lr.fit(X_train_lda, y_train)
    preds = lr.predict(X_test_lda)
    
    return accuracy_score(y_test, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"\nBest Optuna Parameters: {study.best_params}")
print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")

# 4. TRAIN FINAL CLASSIFIER
print("\n--- Training Final Lightweight Classifier ---")
final_lr = LogisticRegression(**study.best_params, max_iter=5000, random_state=42)
final_lr.fit(X_train_lda, y_train)

preds = final_lr.predict(X_test_lda)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 5. EXPORT PURE PYTHON ARRAYS FOR EDGE INFERENCE
print("\n--- Exporting LDA and Classifier Parameters to model_data_lr.py ---")

# Extract LDA parameters
lda_xbar = lda.xbar_.tolist()         # Shape: (39,) -> The mean of each feature
lda_scalings = lda.scalings_.tolist() # Shape: (39, 3) -> The transformation matrix

# Extract Logistic Regression parameters
lr_coef = final_lr.coef_.tolist()           # Shape: (4, 3) -> Weights for the 3 super-features
lr_intercept = final_lr.intercept_.tolist() # Shape: (4,) -> Bias for each class

output_file = os.path.join(OUTPUT_PATH, 'model_data_lr.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated LDA + Logistic Regression Export\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    # Write LDA arrays
    f.write("LDA_XBAR = [\n")
    f.write(f"    {[round(float(val), 6) for val in lda_xbar]}\n")
    f.write("]\n\n")
    
    f.write("LDA_SCALINGS = [\n")
    for row in lda_scalings:
        f.write(f"    {[round(float(val), 6) for val in row]},\n")
    f.write("]\n\n")
    
    # Write LR arrays
    f.write("LR_COEF = [\n")
    for class_weights in lr_coef:
        f.write(f"    {[round(float(w), 6) for w in class_weights]},\n")
    f.write("]\n\n")
    
    f.write("LR_INTERCEPT = [\n")
    f.write(f"    {[round(float(i), 6) for i in lr_intercept]}\n")
    f.write("]\n")

print(f"Pipeline successfully exported to: {output_file}")

[I 2026-06-30 17:05:29,578] A new study created in memory with name: no-name-ba7224a2-42ce-4f74-91d4-f60362e41cfa


[I 2026-06-30 17:05:29,594] Trial 0 finished with value: 0.7773144286905754 and parameters: {'C': 0.2715442248404849}. Best is trial 0 with value: 0.7773144286905754.
[I 2026-06-30 17:05:29,607] Trial 1 finished with value: 0.7781484570475397 and parameters: {'C': 0.5878136026405019}. Best is trial 1 with value: 0.7781484570475397.
[I 2026-06-30 17:05:29,619] Trial 2 finished with value: 0.7773144286905754 and parameters: {'C': 1.704888891340409}. Best is trial 1 with value: 0.7781484570475397.
[I 2026-06-30 17:05:29,631] Trial 3 finished with value: 0.7773144286905754 and parameters: {'C': 14.715221777998774}. Best is trial 1 with value: 0.7781484570475397.
[I 2026-06-30 17:05:29,641] Trial 4 finished with value: 0.7814845704753962 and parameters: {'C': 0.009864865176348103}. Best is trial 4 with value: 0.7814845704753962.
[I 2026-06-30 17:05:29,653] Trial 5 finished with value: 0.7773144286905754 and parameters: {'C': 71.09174961458093}. Best is trial 4 with value: 0.7814845704753962

Loaded Data -> X shape: (5995, 39), y shape: (5995,)

--- Running LDA to reduce 39 features to 3 Super-Features ---
New Training Data Shape after LDA: (4796, 3)

--- Starting Optuna optimization for Logistic Regression ---


[I 2026-06-30 17:05:29,755] Trial 14 finished with value: 0.7831526271893244 and parameters: {'C': 0.0012742555894306184}. Best is trial 6 with value: 0.7848206839032527.
[I 2026-06-30 17:05:29,765] Trial 15 finished with value: 0.7814845704753962 and parameters: {'C': 0.0007881675601853704}. Best is trial 6 with value: 0.7848206839032527.
[I 2026-06-30 17:05:29,778] Trial 16 finished with value: 0.7789824854045038 and parameters: {'C': 0.046899144833490544}. Best is trial 6 with value: 0.7848206839032527.
[I 2026-06-30 17:05:29,790] Trial 17 finished with value: 0.7839866555462885 and parameters: {'C': 0.00199284261315977}. Best is trial 6 with value: 0.7848206839032527.
[I 2026-06-30 17:05:29,804] Trial 18 finished with value: 0.7789824854045038 and parameters: {'C': 0.06175019661317076}. Best is trial 6 with value: 0.7848206839032527.
[I 2026-06-30 17:05:29,815] Trial 19 finished with value: 0.780650542118432 and parameters: {'C': 0.0005616158182245829}. Best is trial 6 with value: 


Best Optuna Parameters: {'C': 0.00010723606365180364}
Best Accuracy during CV: 78.65%

--- Training Final Lightweight Classifier ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.81      0.83      0.82       299
         off       0.77      0.82      0.79       300
        stop       0.85      0.80      0.83       300
     unknown       0.72      0.70      0.71       300

    accuracy                           0.79      1199
   macro avg       0.79      0.79      0.79      1199
weighted avg       0.79      0.79      0.79      1199


--- Exporting LDA and Classifier Parameters to model_data_lr.py ---
Pipeline successfully exported to: ../edge_mcu\model_data_lr.py


In [111]:
import sys
import os

sys.path.append(os.path.abspath('../edge_mcu'))

import model_data_lr as model_data

def predict_audio_class(mfcc_features):
    """
    Two-stage inference:
    1. LDA Transform: Reduces 39 features to 3.
    2. Logistic Regression: Predicts the class based on the 3 features.
    """
    num_original_features = len(model_data.LDA_XBAR[0])
    num_super_features = len(model_data.LDA_SCALINGS[0])
    num_classes = len(model_data.CLASSES)
    
    # --- STAGE 1: LDA Transformation ---
    # Step 1A: Center the data (subtract the mean)
    centered_features = [0.0] * num_original_features
    for i in range(num_original_features):
        centered_features[i] = mfcc_features[i] - model_data.LDA_XBAR[0][i]
        
    # Step 1B: Matrix multiplication (Centered Data * Scalings)
    lda_features = [0.0] * num_super_features
    for i in range(num_super_features):
        sum_val = 0.0
        for j in range(num_original_features):
            sum_val += centered_features[j] * model_data.LDA_SCALINGS[j][i]
        lda_features[i] = sum_val
        
    # --- STAGE 2: Logistic Regression ---
    class_scores = [0.0] * num_classes
    
    for c in range(num_classes):
        # Start with the bias/intercept for this class
        score = model_data.LR_INTERCEPT[0][c]
        
        # Dot product: Super-features * Class Weights
        for i in range(num_super_features):
            score += lda_features[i] * model_data.LR_COEF[c][i]
            
        class_scores[c] = score
        
    # Find the index of the highest score (Argmax)
    best_class = 0
    max_score = class_scores[0]
    for i in range(1, num_classes):
        if class_scores[i] > max_score:
            max_score = class_scores[i]
            best_class = i
            
    return best_class

# --- Example Usage on Board ---
# features = [1.2, -0.5, 3.4, ...] # The 39 MFCC features extracted from the mic
# result_idx = predict_audio_class(features)
# print("Predicted Word:", model_data.CLASSES[result_idx])

import random
for j in range(10):
    i = random.randint(0, len(y_test))
    features = X_test[i]
    result_idx = predict_audio_class(features)
    print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i] + 1)

Predicted Word: stop and actuall is: 3
Predicted Word: on and actuall is: 2
Predicted Word: off and actuall is: 2
Predicted Word: on and actuall is: 1
Predicted Word: stop and actuall is: 3
Predicted Word: stop and actuall is: 3
Predicted Word: off and actuall is: 2
Predicted Word: off and actuall is: 4
Predicted Word: off and actuall is: 2
Predicted Word: unknown and actuall is: 3


LDA + Logistic regression
Targeted training

In [112]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu/'

# Ensure the Output directory exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# 1. Load the FULL 39-feature dataset
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. STAGE 1 & 2: Optuna Optimization for LDA & Logistic Regression combined
print("\n--- Starting Optuna optimization with Focus on Targets ---")
def objective(trial):
    # --- LDA PARAMETERS ---
    lda_shrinkage = trial.suggest_float('lda_shrinkage', 0.0, 1.0)
    lda = LinearDiscriminantAnalysis(solver='eigen', shrinkage=lda_shrinkage, n_components=3)
    
    X_train_lda = lda.fit_transform(X_train, y_train)
    X_test_lda = lda.transform(X_test)
    
    # --- LOGISTIC REGRESSION PARAMETERS ---
    c_val = trial.suggest_float('C', 1e-4, 1e2, log=True)
    
    # --- CLASS PRIORITIZATION ---
    target_weight = trial.suggest_float('target_weight', 1.0, 5.0)
    class_weights = {0: target_weight, 1: target_weight, 2: target_weight, 3: 1.0}
    
    lr = LogisticRegression(C=c_val, class_weight=class_weights, max_iter=5000, random_state=42)
    lr.fit(X_train_lda, y_train)
    preds = lr.predict(X_test_lda)
    
    return accuracy_score(y_test, preds)

# Run optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"\nBest Optuna Parameters: {study.best_params}")
print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")

# 3. TRAIN FINAL PIPELINE WITH BEST PARAMETERS
print("\n--- Training Final Lightweight & Focused Classifier ---")

best_shrinkage = study.best_params['lda_shrinkage']
best_C = study.best_params['C']
best_target_weight = study.best_params['target_weight']

# Train Final LDA (using 'eigen')
final_lda = LinearDiscriminantAnalysis(solver='eigen', shrinkage=best_shrinkage, n_components=3)
X_train_lda_final = final_lda.fit_transform(X_train, y_train)
X_test_lda_final = final_lda.transform(X_test)

# Train Final Logistic Regression
final_weights = {0: best_target_weight, 1: best_target_weight, 2: best_target_weight, 3: 1.0}
final_lr = LogisticRegression(C=best_C, class_weight=final_weights, max_iter=5000, random_state=42)
final_lr.fit(X_train_lda_final, y_train)

# Evaluate
preds = final_lr.predict(X_test_lda_final)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 4. EXPORT PURE PYTHON ARRAYS FOR EDGE INFERENCE
print("\n--- Exporting LDA and Classifier Parameters ---")

# Extract LDA parameters (No xbar_ needed for 'eigen' solver!)
lda_scalings = final_lda.scalings_.tolist() 

# Extract Logistic Regression parameters
lr_coef = final_lr.coef_.tolist()           
lr_intercept = final_lr.intercept_.tolist() 

output_file = os.path.join(OUTPUT_PATH, 'model_data_lr.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Robust LDA (Eigen) + Logistic Regression Export\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    f.write("LDA_SCALINGS = [\n")
    for row in lda_scalings:
        f.write(f"    {[round(float(val), 6) for val in row]},\n")
    f.write("]\n\n")
    
    f.write("LR_COEF = [\n")
    for class_weights in lr_coef:
        f.write(f"    {[round(float(w), 6) for w in class_weights]},\n")
    f.write("]\n\n")
    
    f.write("LR_INTERCEPT = [\n")
    f.write(f"    {[round(float(i), 6) for i in lr_intercept]}\n")
    f.write("]\n")

print(f"Pipeline successfully exported to: {output_file}")

[I 2026-06-30 17:05:33,906] A new study created in memory with name: no-name-41e58485-b3ee-4eb1-bb88-5f0d6395219a
[I 2026-06-30 17:05:33,946] Trial 0 finished with value: 0.7356130108423686 and parameters: {'lda_shrinkage': 0.2627901272707851, 'C': 32.75478872236083, 'target_weight': 3.3456909880904986}. Best is trial 0 with value: 0.7356130108423686.
[I 2026-06-30 17:05:33,980] Trial 1 finished with value: 0.7089241034195163 and parameters: {'lda_shrinkage': 0.2247783098285112, 'C': 0.003334509336343096, 'target_weight': 3.4193028982579565}. Best is trial 0 with value: 0.7356130108423686.
[I 2026-06-30 17:05:34,022] Trial 2 finished with value: 0.6897414512093412 and parameters: {'lda_shrinkage': 0.5661499251951847, 'C': 42.08903225215605, 'target_weight': 4.065726025140012}. Best is trial 0 with value: 0.7356130108423686.
[I 2026-06-30 17:05:34,063] Trial 3 finished with value: 0.6905754795663053 and parameters: {'lda_shrinkage': 0.49232679435521387, 'C': 4.120650115453792, 'target_w

Loaded Data -> X shape: (5995, 39), y shape: (5995,)

--- Starting Optuna optimization with Focus on Targets ---


[I 2026-06-30 17:05:34,119] Trial 4 finished with value: 0.700583819849875 and parameters: {'lda_shrinkage': 0.8556637736174243, 'C': 0.4245007946257436, 'target_weight': 1.6022196331340117}. Best is trial 0 with value: 0.7356130108423686.
[I 2026-06-30 17:05:34,194] Trial 5 finished with value: 0.6146788990825688 and parameters: {'lda_shrinkage': 0.707133847421962, 'C': 0.0003190022784989144, 'target_weight': 1.887000989793886}. Best is trial 0 with value: 0.7356130108423686.
[I 2026-06-30 17:05:34,278] Trial 6 finished with value: 0.6480400333611342 and parameters: {'lda_shrinkage': 0.8984509839612269, 'C': 5.787984081053865, 'target_weight': 4.1773179660174105}. Best is trial 0 with value: 0.7356130108423686.
[I 2026-06-30 17:05:34,318] Trial 7 finished with value: 0.6413678065054211 and parameters: {'lda_shrinkage': 0.20238397726366675, 'C': 0.00012251328602717589, 'target_weight': 4.696676974245686}. Best is trial 0 with value: 0.7356130108423686.
[I 2026-06-30 17:05:34,365] Trial


Best Optuna Parameters: {'lda_shrinkage': 0.10422304392247966, 'C': 86.59055107789042, 'target_weight': 1.2357301231084954}
Best Accuracy during CV: 78.23%

--- Training Final Lightweight & Focused Classifier ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.79      0.83      0.81       299
         off       0.79      0.81      0.80       300
        stop       0.83      0.80      0.81       300
     unknown       0.72      0.69      0.71       300

    accuracy                           0.78      1199
   macro avg       0.78      0.78      0.78      1199
weighted avg       0.78      0.78      0.78      1199


--- Exporting LDA and Classifier Parameters ---
Pipeline successfully exported to: ../edge_mcu/model_data_lr.py


In [113]:
import model_data_lr

def predict_audio_class(mfcc_features):
    """
    Super-optimized Inference Engine for MicroPython (LDA Eigen + Logistic Regression).
    File size: < 3KB | Execution time: < 2ms
    """
    num_original_features = 39 # 13 MFCCs * 3 Time Frames
    num_super_features = 3     # Reduced dimensions by LDA
    num_classes = len(model_data_lr.CLASSES)
    
    # --- STAGE 1: LDA Transformation (Direct Matrix Multiplication) ---
    # Since we used 'eigen' solver, we directly multiply raw features by SCALINGS
    lda_features = [0.0] * num_super_features
    for i in range(num_super_features):
        sum_val = 0.0
        for j in range(num_original_features):
            sum_val += mfcc_features[j] * model_data_lr.LDA_SCALINGS[j][i]
        lda_features[i] = sum_val
        
    # --- STAGE 2: Logistic Regression Inference ---
    class_scores = [0.0] * num_classes
    for c in range(num_classes):
        # Start with the bias/intercept for this class
        score = model_data_lr.LR_INTERCEPT[0][c]
        
        # Dot product: Super-features * Class Weights
        for i in range(num_super_features):
            score += lda_features[i] * model_data_lr.LR_COEF[c][i]
            
        class_scores[c] = score
        
    # Find the index of the highest score (Argmax)
    best_class = 0
    max_score = class_scores[0]
    for i in range(1, num_classes):
        if class_scores[i] > max_score:
            max_score = class_scores[i]
            best_class = i
            
    return best_class

# --- Example Usage on Board ---
# features = [1.2, -0.5, 3.4, ...] # 39 features extracted from mic (VAD applied)
# result_idx = predict_audio_class(features)
# print("Predicted Word:", model_data_lr.CLASSES[result_idx])

import random
for j in range(10):
    i = random.randint(0, len(y_test))
    features = X_test[i]
    result_idx = predict_audio_class(features)
    print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i] + 1)

Predicted Word: stop and actuall is: 3
Predicted Word: on and actuall is: 2
Predicted Word: stop and actuall is: 3
Predicted Word: stop and actuall is: 2
Predicted Word: off and actuall is: 2
Predicted Word: unknown and actuall is: 4
Predicted Word: on and actuall is: 3
Predicted Word: on and actuall is: 4
Predicted Word: on and actuall is: 1
Predicted Word: on and actuall is: 3
